## building RAG System using langchain and chromaDB


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import pandas as pd
from typing import List, Dict, Any

In [6]:
sample_text = """
Python Programming Introduction

Python is a high-level programming language known for its simplicity and readability. 
It was created by Guido van Rossum and released in 1991.

Key Features:
- Easy to learn and use
- Large community support
- Cross-platform compatibility

Python is widely used in web development, data science, machine learning, and automation.


Machine Learning Basics

Machine learning is a branch of artificial intelligence that allows computers to learn 
from data without being explicitly programmed.

Types of Machine Learning:
1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning

Applications include image recognition, recommendation systems, and fraud detection.


Data Science Fundamentals

Data science combines statistics, programming, and domain knowledge to extract insights 
from data.

Key Steps:
- Data collection
- Data cleaning
- Data analysis
- Visualization

Tools used include Python, R, and SQL.

"""

In [23]:
sample_text

'\nPython Programming Introduction\n\nPython is a high-level programming language known for its simplicity and readability. \nIt was created by Guido van Rossum and released in 1991.\n\nKey Features:\n- Easy to learn and use\n- Large community support\n- Cross-platform compatibility\n\nPython is widely used in web development, data science, machine learning, and automation.\n\n\nMachine Learning Basics\n\nMachine learning is a branch of artificial intelligence that allows computers to learn \nfrom data without being explicitly programmed.\n\nTypes of Machine Learning:\n1. Supervised Learning\n2. Unsupervised Learning\n3. Reinforcement Learning\n\nApplications include image recognition, recommendation systems, and fraud detection.\n\n\nData Science Fundamentals\n\nData science combines statistics, programming, and domain knowledge to extract insights \nfrom data.\n\nKey Steps:\n- Data collection\n- Data cleaning\n- Data analysis\n- Visualization\n\nTools used include Python, R, and SQL.

In [25]:
with open("../data/python.txt", "w") as f:
    f.write(sample_text)

### Document loader

In [7]:
loader=TextLoader("../data/python.txt", encoding="utf-8")
documents = loader.load()
documents
for doc in documents:
    print(doc.page_content)


Python Programming Introduction

Python is a high-level programming language known for its simplicity and readability. 
It was created by Guido van Rossum and released in 1991.

Key Features:
- Easy to learn and use
- Large community support
- Cross-platform compatibility

Python is widely used in web development, data science, machine learning, and automation.


Machine Learning Basics

Machine learning is a branch of artificial intelligence that allows computers to learn 
from data without being explicitly programmed.

Types of Machine Learning:
1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning

Applications include image recognition, recommendation systems, and fraud detection.


Data Science Fundamentals

Data science combines statistics, programming, and domain knowledge to extract insights 
from data.

Key Steps:
- Data collection
- Data cleaning
- Data analysis
- Visualization

Tools used include Python, R, and SQL.




### Document splitting

In [8]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=100,
 chunk_overlap=20,
 separators=["\n\n", "\n", " ", ""])
docs = doc_splitter.split_documents(documents)
for doc in docs:
    print(doc.page_content)
    print("\n---\n")


Python Programming Introduction

---

Python is a high-level programming language known for its simplicity and readability.

---

It was created by Guido van Rossum and released in 1991.

---

Key Features:
- Easy to learn and use
- Large community support
- Cross-platform compatibility

---

Python is widely used in web development, data science, machine learning, and automation.

---

Machine Learning Basics

---

Machine learning is a branch of artificial intelligence that allows computers to learn

---

from data without being explicitly programmed.

---

Types of Machine Learning:
1. Supervised Learning
2. Unsupervised Learning

---

3. Reinforcement Learning

---

Applications include image recognition, recommendation systems, and fraud detection.

---

Data Science Fundamentals

---

Data science combines statistics, programming, and domain knowledge to extract insights

---

from data.

---

Key Steps:
- Data collection
- Data cleaning
- Data analysis
- Visualization

---

Tool

### chunks embedding

In [9]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embedding_model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1270.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### initilize chromaDB vectorDB and store chunks

In [10]:

persist_directory="./chroma_db"
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="rag_system",
    persist_directory=persist_directory
)
print(f"vectorstore: {vectorstore._collection.count()}")
print(f"persisted to {persist_directory}")

vectorstore: 128
persisted to ./chroma_db


### test similarty search

In [10]:
query = "What are the key features of Python programming?"
similarity_search_results = vectorstore.similarity_search(query, k=3)
similarity_search_results

[Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.'),
 Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.'),
 Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.')]

### inilize llm , RAG Chain, promot templeate ,query RAg system

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [12]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [13]:
from langchain.chat_models.base import init_chat_model
llm = init_chat_model("groq:llama-3.3-70b-versatile")

In [14]:
llm.invoke("what is AI")

AIMessage(content='**Artificial Intelligence (AI)**: Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:\n\n1. **Learning**: AI systems can learn from data and improve their performance over time.\n2. **Problem-solving**: AI systems can analyze problems and develop solutions.\n3. **Reasoning**: AI systems can draw inferences and make decisions based on available data.\n4. **Perception**: AI systems can interpret and understand data from sensors, such as images and speech.\n\n**Key Characteristics of AI**:\n\n1. **Intelligence**: AI systems can perform tasks that would typically require human intelligence.\n2. **Autonomy**: AI systems can operate independently, making decisions without human intervention.\n3. **Adaptability**: AI systems can adapt to changing environments and learn from experience.\n\n**Types of AI**:\n\n1. **Narrow or Weak AI**: Designed to perform a specific task, such as imag

### modern RAG chain

In [15]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [14]:
retrieval=vectorstore.as_retriever(
    search_kwargs={"k": 3},
    
)

In [17]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """You are a helpful assistant that provides concise answers to questions based on the retrieved documents.
If you don't know the answer, say you don't know. Always use all the retrieved documents to provide the best possible answer.
context: {context}
"""
prompt = ChatPromptTemplate.from_messages(
   [("system", system_prompt), ("human", "{input}")]
)

In [15]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
combine_docs_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
combine_docs_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant that provides concise answers to questions based on the retrieved documents.\nIf you don't know the answer, say you don't know. Always use all the retrieved documents to provide the best possible answer.\ncontext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, '

In [16]:
from langchain_classic.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(
    retriever=retrieval,
    combine_docs_chain=combine_docs_chain
)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D3D97E1550>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant that provides concise answers to questions based on the retrieved documents.\nIf you don't know t

In [20]:
rag_chain.invoke({"input": "What are the key features of Python programming?"})

{'input': 'What are the key features of Python programming?',
 'context': [Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.'),
  Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.'),
  Document(metadata={'source': '../data/python.txt'}, page_content='Python is a high-level programming language known for its simplicity and readability.')],
 'answer': 'The key features of Python programming include its simplicity and readability.'}

In [21]:
rag_chain.invoke({"input":"who is the best player in the world?"})

{'input': 'who is the best player in the world?',
 'context': [Document(metadata={'source': '../data/python.txt'}, page_content='It was created by Guido van Rossum and released in 1991.'),
  Document(metadata={'source': '../data/python.txt'}, page_content='It was created by Guido van Rossum and released in 1991.'),
  Document(metadata={'source': '../data/python.txt'}, page_content='It was created by Guido van Rossum and released in 1991.')],
 'answer': "I don't know. The provided documents only mention Guido van Rossum and the release of Python in 1991, but they do not contain information about the best player in the world."}

In [22]:
response=rag_chain.invoke({"input":"what type of machine learning ?"})
print(response["answer"])

There are two main types: 
1. Supervised Learning
2. Unsupervised Learning


In [24]:
response=rag_chain.invoke({"input":"ما هي اساسيات التعلم الآلي؟"})
print(response["answer"])

أساسيات التعلم الآلي (Machine Learning) هي مجموعة من المفاهيم والتقنيات التي تمكن الأنظمة من التعلم من البيانات وتحسين أدائها دون الحاجة إلى برمجة صريحة. وتشمل هذه الأساسيات:

1. **التعلم من البيانات**: التعلم الآلي يعتمد على البيانات التي يتم جمعها وتحليلها لتدريب النماذج. وتستخدم هذه البيانات لتعليم النظام على التعرف على الأنماط والقوانين التي تحكم السلوك أو الظاهرة المراد دراستها.

2. **النماذج الرياضية**: التعلم الآلي يستخدم نماذج رياضية لممارسة التعلم والتنبؤ. هذه النماذج يمكن أن تكون خطية أو غير خطية، وتتضمن معادلات ومعادلات تفاضلية وغيرها من الأدوات الرياضية.

3. **الخوارزميات**: الخوارزميات هي مجموعات من الإجراءات التي يتم تنفيذها على البيانات لتدريب النماذج. وتشمل هذه الخوارزميات خوارزميات التعلم الإشرافي (Supervised Learning) والتعلم غير الإشرافي (Unsupervised Learning) والتعلم بالتعزيز (Reinforcement Learning).

4. **التحليل والتنبؤ**: بعد تدريب النماذج، يتم استخدامها للتنبؤ ببيانات جديدة. وهذا يتطلب تحليل البيانات المدخلة وتطبيق النموذج عليها لتحديد النتائج المتوقعة.

5. **

In [25]:
response=rag_chain.invoke({"input":"ايه انواع التعليم الآلي؟"})
print(response["answer"])

هناك ثلاثة أنواع رئيسية من التعليم الآلي:

1. **التعلم الآلي الخاضع للإشراف (Supervised Learning)**: في هذا النوع، يتم تدريب النموذج على بيانات مصنفة مسبقًا، حيث يتم توفير الإجابات الصحيحة للنموذج. الهدف هو تعلم النموذج من هذه البيانات لتقديم تنبؤات دقيقة على بيانات جديدة.

2. **التعلم الآلي غير الخاضع للإشراف (Unsupervised Learning)**: في هذا النوع، لا يتم توفير إجابات صحيحة للنموذج. بدلاً من ذلك، يتم تدريب النموذج علىデータ غير مصنفة لاكتشاف الأنماط والت聚يرات فيها.

3. **التعلم الآلي بالتدريب الذاتي (Reinforcement Learning)**: في هذا النوع، يتم تدريب النموذج من خلال التجارب والتفاعل مع البيئة. يتلقى النموذج مكافآت أو عقوبات بناءً على أفعالة، ويتعلم من هذه التفاعلات لتحسين أدائه وتحقيق الأهداف المحددة.


### create rag chain alternative usin LCEL

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [3]:
# create custom prompt
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """You are a helpful assistant that provides concise answers to questions based on the retrieved documents.
If you don't know the answer, say you don't know. Always use all the retrieved documents to provide the best possible answer.
context: {context}
"""
prompt = ChatPromptTemplate.from_messages(
   [("system", system_prompt), ("human", "{input}")]
)

In [18]:

retrieval

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D3D97E1550>, search_kwargs={'k': 3})

In [19]:
## format docs for the prompt
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [21]:
rag_chain_LCEL = (
    {"context": retrieval | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser())

In [107]:
response = rag_chain_LCEL.invoke("what are the key features of Python programming?")
print(response)

The key features of Python programming include its simplicity and readability. However, based on general knowledge, some additional key features are:

1. High-level language: Python is a high-level language, meaning it abstracts away many low-level details, making it easier to focus on the logic of the program.
2. Easy to learn: Python has a simple syntax and is relatively easy to learn, making it a great language for beginners.
3. Object-oriented: Python supports object-oriented programming (OOP) concepts, such as classes, objects, and inheritance.
4. Large standard library: Python has a vast collection of libraries and modules that make it easy to perform various tasks, such as file I/O, networking, and data analysis.
5. Cross-platform: Python can run on multiple operating systems, including Windows, macOS, and Linux.
6. Dynamic typing: Python is dynamically typed, which means you don't need to declare the data type of a variable before using it.
7. Extensive community: Python has a 

### add new data to vector store

In [26]:
new_doc="""Rag system is a powerful tool for building question-answering applications. It combines retrieval and generation to provide accurate and relevant answers based on a given context. The key features of a rag system include:
1. Retrieval: The system retrieves relevant documents or information from a knowledge base or database based on the user's query.
2. Generation: The system generates a response based on the retrieved information, often using natural language generation techniques.
3. Contextual Understanding: The system understands the context of the query and the retrieved information to provide accurate answers.
4. Scalability: Rag systems can handle large volumes of data and queries efficiently.   
"""

In [27]:
new_docment = Document(page_content=new_doc, metadata={"source": "new_doc"})

In [29]:
# split the new document into chunks
new_docs = doc_splitter.split_documents([new_docment])
new_docs

[Document(metadata={'source': 'new_doc'}, page_content='Rag system is a powerful tool for building question-answering applications. It combines retrieval'),
 Document(metadata={'source': 'new_doc'}, page_content='combines retrieval and generation to provide accurate and relevant answers based on a given'),
 Document(metadata={'source': 'new_doc'}, page_content='based on a given context. The key features of a rag system include:'),
 Document(metadata={'source': 'new_doc'}, page_content='1. Retrieval: The system retrieves relevant documents or information from a knowledge base or'),
 Document(metadata={'source': 'new_doc'}, page_content="a knowledge base or database based on the user's query."),
 Document(metadata={'source': 'new_doc'}, page_content='2. Generation: The system generates a response based on the retrieved information, often using'),
 Document(metadata={'source': 'new_doc'}, page_content='often using natural language generation techniques.'),
 Document(metadata={'source': 'n

In [30]:
vectorstore.add_documents(new_docs)

['99684a87-6ff4-4cdb-b4ed-7b176bd72669',
 'deaeb920-1c42-44f5-a96a-695eb0645715',
 '4e4da967-d53d-4f85-af62-6d9096b7fc77',
 '144ece1b-d620-4d39-9fca-55e8b7667d6c',
 'da841cac-9672-4aa5-979b-7e744d842734',
 '8a8c24c7-bd19-471f-b94b-3f376ba84016',
 '7b2397aa-7e43-4c97-a509-95e305f78dae',
 '57416efe-f709-4881-ba88-1962566bb411',
 '00f7c454-1f2a-431d-adaa-f7d71ed8dd28',
 '1eb132b9-9788-4540-8cf1-1cf0022ff671']

In [31]:
new_query = "what are the key features of a rag system?"
response = rag_chain_LCEL.invoke(new_query)
print(response)

Based on the given context, one of the key features of a Rag system is:

1. Scalability: Rag systems can handle large volumes of data and queries efficiently.

However, it is mentioned that there are multiple key features, but only scalability is specified in the provided context. Therefore, I don't know the other key features of a Rag system.


### advanced rag techniques - convesational memory

In [34]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import ChatMessage, HumanMessage,AIMessage,SystemMessage

In [59]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])



In [60]:
history_aware=create_history_aware_retriever(
llm,retrieval,prompt

)
history_aware

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D3D97E1550>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMess

In [61]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [67]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a branch of artificial intelligence. It allows computers to learn, enabling them to improve their performance on tasks over time. This is achieved through various algorithms and techniques that enable computers to learn from data.


In [68]:
from langchain_core.messages import ChatMessage, HumanMessage,AIMessage,SystemMessage
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [70]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a branch of artificial intelligence. It allows computers to learn, enabling them to improve their performance on tasks over time. This is achieved through various algorithms and techniques that enable computers to learn from data.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [72]:
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?" ,
     "context": result1['answer']  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a branch of artificial intelligence. It allows computers to learn, enabling them to improve their performance on tasks over time. This is achieved through various algorithms and techniques that enable computers to learn from data.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': '../data/python.txt'}, page_content='3. Reinforcement Learning'),
  Document(metadata={'source': '../data/python.txt'}, page_content='3. Reinforcement Learning'),
  Document(metadata={'source': '../data/python.txt'}, page_content='3. Reinforcement Learning')],
 'answer': 'The main types of machine learning are Supervised Learning, Unsupervised Learning, and Reinforcement Learning. These categories differ in the way they learn from data and the ty

In [110]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.messages import HumanMessage
# ── 1. مثال للمحادثة السابقة
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# ── 2. Prompt لإعادة صياغة السؤال (History-aware)
rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", "أنت مساعد الجامعة الذكي. استخدم تاريخ المحادثة لإعادة صياغة السؤال بطريقة واضحة للبحث."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# ── 3. History-aware retriever
# تأكد من أنك عرفت llm و retrieval مسبقاً
# llm = ChatOpenAI(...)
# retrieval = vectorstore.as_retriever()

history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retrieval,
    prompt=rephrase_prompt
)

qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

# ── 4. Prompt النهائي للإجابة
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── 5. LCEL Chain النهائي - التصحيح
rag_chain_lcel = (
    {
        "context": history_aware_retriever | format_docs,
        "input": lambda x: x["input"],           # استخراج input من المدخلات
        "chat_history": lambda x: x["chat_history"]  # استخراج chat_history
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# المحادثة
chat_history = []

# First question
result1 = rag_chain_lcel.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})

print(f"Q: What is machine learning?")
print(f"A: {result1}")  # التصحيح: result1 نص وليس قاموس

# تحديث المحادثة للسؤال التالي
chat_history.append(HumanMessage(content="What is machine learning?"))
chat_history.append(result1)  # أضف الإجابة (تأكد من نوعها)

# Second question with history
result2 = rag_chain_lcel.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"
})

print(f"\nQ: What are its main types?")
print(f"A: {result2}")

Q: What is machine learning?
A: Machine learning is a branch of artificial intelligence that allows computers to learn. It enables computers to acquire knowledge and improve their performance on a task without being explicitly programmed. This allows them to make predictions, classify data, and make decisions based on the data they have learned from.

Q: What are its main types?
A: The main types of machine learning are Supervised Learning and Unsupervised Learning. Supervised Learning involves training a model on labeled data, while Unsupervised Learning involves training a model on unlabeled data. These two types are the primary categories of machine learning.
